# Sesión práctica: LLMs fundacionales con OpenAI, Google Gemini y LangChain


En esta sesión vamos a trabajar con modelos fundacionales consumidos vía API. El objetivo no es entrenar un modelo desde cero, sino aprender a usar modelos ya entrenados como piezas de software: configurar credenciales, elegir modelos, enviar mensajes, controlar su comportamiento, gestionar conversaciones, medir tokens, generar respuestas en streaming y pedir salidas estructuradas.

Vamos a ver los conceptos de forma paralela en tres niveles:

1. **OpenAI API**, como ejemplo de proveedor directo.
2. **Google Gemini API**, como segundo proveedor directo con una sintaxis distinta pero una lógica muy similar.
3. **LangChain**, como capa de abstracción que permite trabajar con distintos proveedores siguiendo patrones comunes.

La idea importante de la sesión es que el concepto no cambia tanto como la sintaxis: casi siempre tendremos un cliente o wrapper, un identificador de modelo, un input, una configuración, una respuesta y algún mecanismo para extraer el contenido útil.


## 0. Requisitos previos


* Python >= 3.9.
* Cuenta en [OpenAI](https://platform.openai.com/) y en [Google AI Studio](https://ai.google.dev/) con claves API vigentes.
* Extensión **Jupyter** en VS Code.
* Un entorno virtual activo para aislar dependencias del resto de proyectos.

> **Consejo:**
>
> Guarda tus claves en variables de entorno (`OPENAI_API_KEY`, `GOOGLE_API_KEY`) o en un archivo `.env` que no subas a control de versiones. Esto debes hacerlo antes de empezar a ejecutar este notebook. Si ya has empezado a trabajar y acabas de crear el archivo `.env`, reinicia el kernel antes de continuar para evitar estados inconsistentes.
>
> La opción recomendada para esta sesión es definir dichas variables de entorno en un fichero `.env`. Tienes disponible un fichero `.env.template` que indica la forma esperada del archivo:
>
> 1. Crea un fichero llamado `.env` en la misma carpeta donde se ubica este notebook.
> 2. Copia el contenido de `.env.template` y pégalo en tu fichero `.env`.
> 3. Sustituye el valor de `OPENAI_API_KEY` y `GOOGLE_API_KEY` por tus respectivas API keys.
>
> Si prefieres guardar las claves como variables de entorno del sistema:
>
> * **MacOS/Linux:** utiliza en el terminal `export OPENAI_API_KEY=<tu API key de OpenAI>` y `export GOOGLE_API_KEY=<tu API key de Google>`.
> * **Windows:** utiliza en PowerShell `setx OPENAI_API_KEY "tu API key de OpenAI"` y `setx GOOGLE_API_KEY "tu API key de Google"`.


### Generación de la API key de OpenAI

Para generar la API key de OpenAI, accede a [platform.openai.com](https://platform.openai.com/docs/overview) e inicia sesión. Una vez hayas iniciado sesión, dirígete a la sección *Dashboard*.

<img src="./img/openai_dashboard.png" alt="Sección 'Dashboard' dentro del Platform de OpenAI" style="display:block; margin:auto; margin-bottom:20px;" width="500"/>

Dentro de esta sección, en el menú de navegación de la izquierda, accede a *API keys* y clica en *Create new secret key*.

<img src="./img/openai_api_keys.png" alt="Sección de 'API keys' dentro del Platform de OpenAI" style="display:block; margin:auto; margin-bottom:20px;" width="500"/>

Dale un nombre identificativo y clica en *Create secret key*.

<img src="./img/openai_create_api_key.png" alt="Crear 'API key' dentro del Platform de OpenAI" style="display:block; margin:auto; margin-bottom:20px;" width="500"/>

> Una vez creada tu API key, cópiala y guárdala en un lugar seguro. Una vez hagas click en *Done*, no podrás volver a acceder a esta API key en concreto.


### Generación de la API key de Google

Para crear la API key que nos permita utilizar los modelos de Google, deberemos acceder a la sección de API keys de Google AI Studio. Puedes acceder a ella desde [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey).

<img src="./img/google_api_keys.png" alt="Sección 'API Keys' dentro de Google AI Studio" style="display:block; margin:auto; margin-bottom:20px;" width="500"/>

Aquí, clicaremos en el botón *Crear clave de API*. Deberemos darle un nombre identificativo a nuestra API key y asignarla a un proyecto que ya hubiéramos creado previamente. Si no hemos creado ningún proyecto anteriormente al que queramos asignar dicha API key, clicaremos la opción *Crear proyecto* y seguiremos los pasos que se nos indican.

<img src="./img/google_create_api_key.png" alt="Crear API key en Google AI Studio" style="display:block; margin:auto; margin-bottom:20px;" width="500"/>

<img src="./img/google_create_project.png" alt="Crear proyecto en Google AI Studio" style="display:block; margin:auto; margin-bottom:20px;" width="500"/>

Nótese que, si el proyecto es de nueva creación y no hemos configurado ningún método de facturación previamente, tendremos que hacerlo clicando en *Configurar la facturación* en la columna *Nivel de facturación*.

<img src="./img/google_configure_payment.png" alt="Configurar facturación en Google AI Studio" style="display:block; margin:auto; margin-bottom:20px;" width="500"/>

Una vez creada la API key, la copiaremos y pegaremos en el fichero `.env` de este repositorio o la exportaremos como variable de entorno.


In [ ]:
# NOTE: Si ya has creado tu entorno virtual e instalado las dependencias, puedes omitir esta celda.

# Instalar (o actualizar) las dependencias principales.
# Esta celda se ejecutará una sola vez al inicio del notebook. Reinicia el kernel posteriormente si es necesario.
%pip install -q -U openai google-genai langchain langchain-openai langchain-google-genai langgraph python-dotenv tiktoken pydantic


## 1. Carga de credenciales

A continuación cargaremos las variables de entorno que alojan las API keys tanto de OpenAI como de Google. Estas claves serán utilizadas por los clientes oficiales y también por LangChain, ya que LangChain delega la llamada final en el proveedor correspondiente.

La siguiente celda intenta cargar primero un fichero `.env`. Si las claves no están disponibles tras esa carga, se solicitarán manualmente.


In [ ]:
import os, getpass
from dotenv import load_dotenv

# Intenta cargar las variables desde el archivo .env
dotenv_loaded = load_dotenv()

# Solicita las claves si no están definidas
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google Gemini API Key: ")

# Si después de todo no hay claves, lanza un error
if not os.getenv("OPENAI_API_KEY") or not os.getenv("GOOGLE_API_KEY"):
    raise RuntimeError("No se encontraron las API keys requeridas. Por favor, agrégalas al archivo .env o introdúcelas manualmente.")

print("Claves cargadas en variables de entorno.")


## 2. El patrón común: cliente, modelo, input y respuesta

Antes de entrar en detalles específicos de cada proveedor, conviene fijar el patrón que veremos repetido durante toda la sesión.

Cuando consumimos un LLM vía API, normalmente seguimos estos pasos:

1. Importamos el SDK o wrapper correspondiente.
2. Definimos el modelo que queremos usar.
3. Creamos un cliente o wrapper autenticado.
4. Enviamos un input al modelo.
5. Extraemos de la respuesta el contenido que nos interesa.

La sintaxis cambia entre OpenAI, Gemini y LangChain, pero la idea base es exactamente la misma.


In [ ]:
from openai import OpenAI
from google import genai
from langchain.chat_models import init_chat_model

OPENAI_MODEL = "gpt-4.1-nano"
GOOGLE_MODEL = "gemini-2.5-flash-lite"

openai_client = OpenAI()
gemini_client = genai.Client()

langchain_openai_model = init_chat_model(OPENAI_MODEL, model_provider="openai")
langchain_google_model = init_chat_model(GOOGLE_MODEL, model_provider="google_genai")


### 2.1. Primera llamada a un modelo

Vamos a pedir exactamente la misma tarea en los tres enfoques: traducir al italiano la frase *Hello, how are you?*.

Esto nos permite comparar el flujo mental en cada caso:

| Enfoque | Objeto principal | Método de llamada | Atributo habitual de texto |
|---------|------------------|-------------------|-----------------------------|
| OpenAI | `OpenAI()` | `client.responses.create(...)` | `response.output_text` |
| Gemini | `genai.Client()` | `client.models.generate_content(...)` | `response.text` |
| LangChain | `init_chat_model(...)` | `model.invoke(...)` | `response.content` |

Fíjate en que LangChain no sustituye al proveedor. Lo envuelve. Seguimos usando un modelo de OpenAI o Google, pero con una interfaz más homogénea.


In [ ]:
prompt_text = "Translate the following text to Italian: 'Hello, how are you?'"

openai_response = openai_client.responses.create(
    model=OPENAI_MODEL,
    input=prompt_text,
)

print("OPENAI:")
print(openai_response.output_text)

gemini_response = gemini_client.models.generate_content(
    model=GOOGLE_MODEL,
    contents=prompt_text,
)

print("\nGEMINI:")
print(gemini_response.text)

langchain_openai_response = langchain_openai_model.invoke(prompt_text)
langchain_google_response = langchain_google_model.invoke(prompt_text)

print("\nLANGCHAIN + OPENAI:")
print(langchain_openai_response.content)

print("\nLANGCHAIN + GEMINI:")
print(langchain_google_response.content)


### 2.2. Inspeccionando los objetos de respuesta

Cuando estamos empezando, es tentador quedarse únicamente con el texto final. Sin embargo, en una integración real suele interesarnos inspeccionar el objeto completo, porque ahí aparecen metadatos de uso, identificadores, candidatos, eventos, información de seguridad, tool calls, mensajes parciales y otros detalles útiles.

En OpenAI, para acceder al texto final normalmente usaremos `output_text`. También podemos observar `response.output`, que es una lista. Esta lista puede contener más de un elemento, especialmente si hacemos uso de *function calls*, herramientas externas, búsqueda, recuperación de archivos o código ejecutado por el modelo. Por eso no conviene asumir que la respuesta siempre estará en `output[0].content[0].text`, aunque en un ejemplo simple pueda coincidir.

En Gemini, `response.text` nos da el texto de forma cómoda, pero el objeto completo puede incluir candidatos, metadatos de uso y otros campos. En LangChain, el resultado de `invoke` es normalmente un `AIMessage`, cuyo texto está en `content` y cuyos metadatos aparecen en atributos como `response_metadata` o `usage_metadata`, dependiendo del proveedor y de la versión del wrapper.


In [ ]:
print("OPENAI response.output:")
print(openai_response.output)

print("\nGEMINI response object type:")
print(type(gemini_response))

print("\nLANGCHAIN response object types:")
print(type(langchain_openai_response))
print(type(langchain_google_response))

print("\nLANGCHAIN metadata examples:")
print("OpenAI usage metadata:", getattr(langchain_openai_response, "usage_metadata", None))
print("Gemini usage metadata:", getattr(langchain_google_response, "usage_metadata", None))


## 3. Elección de modelos

La elección de modelo no debería hacerse copiando un nombre cualquiera de un ejemplo. En un proyecto real, escoger modelo es una decisión de producto e ingeniería: afecta a coste, latencia, calidad, privacidad, límites de contexto, modalidad y estabilidad de la aplicación.

En esta sesión usaremos:

* `gpt-4.1-nano` en OpenAI, porque es una opción barata y rápida para practicar llamadas por API.
* `gemini-2.5-flash-lite` en Google, porque también está orientado a bajo coste y baja latencia.

El objetivo de estos modelos en clase no es demostrar el máximo rendimiento posible, sino permitir muchas llamadas pequeñas, rápidas y económicas para aprender los patrones de uso.


### 3.1. Criterios prácticos para elegir modelo

Antes de fijar un modelo, plantéate estas preguntas:

* **Calidad requerida:** ¿necesitas una respuesta muy precisa o basta con una respuesta razonable?
* **Coste:** ¿vas a hacer pocas llamadas o miles/millones de llamadas?
* **Latencia:** ¿el usuario espera una respuesta inmediata o puede esperar unos segundos?
* **Ventana de contexto:** ¿el input cabe en unos pocos mensajes o necesitas procesar documentos largos, historiales o código completo?
* **Modalidades:** ¿solo necesitas texto o también imagen, audio, vídeo o herramientas?
* **Privacidad y cumplimiento:** ¿puedes enviar esos datos a un proveedor externo? ¿estás usando capa gratuita o de pago?
* **Estabilidad:** ¿necesitas un alias que evolucione o un snapshot/versionado estable para producción?

En OpenAI, la forma de entender el catálogo actual no consiste solo en separar rígidamente entre “modelos GPT” y “modelos razonadores”. Los modelos modernos pueden incluir razonamiento configurable mediante parámetros como `reasoning.effort`, de modo que la diferencia práctica está más en el equilibrio entre calidad, coste, latencia, ventana de contexto y capacidades soportadas.

Una forma útil de clasificar modelos de OpenAI es:

* **Modelos frontier:** priorizan calidad, razonamiento y capacidad para tareas complejas. Son una buena opción para análisis profundos, programación, planificación o trabajo profesional con alta exigencia.
* **Modelos mini/nano:** priorizan coste y latencia. Son ideales para ejemplos de clase, prototipos, clasificación simple, extracción de información o tareas de gran volumen.
* **Modelos especializados:** están pensados para modalidades o tareas concretas, como audio, imagen, tiempo real, embeddings, moderación o búsqueda.
* **Modelos legacy:** modelos de generaciones anteriores pueden aparecer en documentación antigua, ejemplos previos o proyectos existentes. Conviene reconocerlos, pero antes de usarlos en un proyecto nuevo hay que comprobar si siguen recomendados, disponibles o deprecados.

Puedes ver el listado completo y actualizado de modelos de OpenAI desde [la documentación oficial](https://platform.openai.com/docs/models).


### 3.2. Catálogo de modelos en Google Gemini

Google también ofrece diferentes familias y variantes de modelos. En proyectos reales, la elección debería seguir los mismos criterios: calidad, coste, latencia, ventana de contexto, modalidad, privacidad y estabilidad.

Google organiza su catálogo en familias como Gemini Pro, Gemini Flash, Gemini Flash-Lite y modelos especializados para audio, imagen, vídeo, embeddings, robótica o herramientas. Algunas generaciones anteriores pueden aparecer como *deprecated* o haber sido sustituidas por variantes más recientes, así que conviene comprobar la documentación antes de fijar un modelo en producción.

Puedes ver todas las variantes que ofrece Google desde [la documentación oficial de modelos](https://ai.google.dev/gemini-api/docs/models). Por otro lado, desde [la página oficial de precios](https://ai.google.dev/gemini-api/docs/pricing) puedes acceder al catálogo de precios y límites.

> Nótese que Google ofrece una capa gratuita en el uso de algunos modelos desde la API para probarlos. En la documentación de precios se indica que, en la capa gratuita, el contenido puede usarse para mejorar sus productos; en la capa de pago, no. Por tanto, no uses información sensible o confidencial en ejemplos de clase ni en cuentas gratuitas.


### 3.3. Qué cambia LangChain en la elección de modelo

LangChain no elimina la necesidad de elegir bien el modelo. Lo que aporta es una interfaz común para instanciar modelos de diferentes proveedores. En lugar de importar manualmente cada SDK y aprender una forma completamente distinta de invocación, podemos inicializar wrappers mediante `init_chat_model`.

Aun así, el nombre del modelo y el proveedor siguen importando. No todos los proveedores soportan exactamente las mismas funcionalidades: salida estructurada, tool calling, streaming, visión, audio, razonamiento configurable, metadatos de uso o determinados roles pueden variar.

Por eso, LangChain debe entenderse como una capa de abstracción útil, no como una garantía de que todas las capacidades serán idénticas entre proveedores.


In [ ]:
print("Modelo OpenAI directo:", OPENAI_MODEL)
print("Modelo Gemini directo:", GOOGLE_MODEL)
print("Wrapper LangChain OpenAI:", type(langchain_openai_model))
print("Wrapper LangChain Gemini:", type(langchain_google_model))


## 4. Instrucciones, roles y configuración de generación

Un modelo fundacional no está entrenado para una única tarea concreta. Puede traducir, resumir, clasificar, escribir código, explicar conceptos, generar borradores, razonar sobre documentos o actuar como asistente conversacional. Por tanto, tenemos que indicarle qué esperamos de él.

La forma más básica sería incluir la instrucción dentro del propio texto:

```text
Translate the following text to Italian: 'Hello, how are you?'
```

Pero en aplicaciones reales suele ser mejor separar:

* Las reglas de comportamiento de la aplicación.
* El input concreto del usuario.
* El historial de mensajes anteriores.
* La configuración técnica de generación.

Esa separación se expresa mediante roles, instrucciones de sistema, objetos de configuración o plantillas de prompt.


### 4.1. Roles e instrucciones en OpenAI, Gemini y LangChain

En OpenAI podemos usar mensajes con roles o el parámetro `instructions`. Los roles principales en la API de Responses son:

| **DEVELOPER** | **USER** | **ASSISTANT** |
|---------------|----------|---------------|
| Instrucciones dadas por el desarrollador de la aplicación. Tienen prioridad frente a los mensajes de usuario. | Inputs o instrucciones del usuario final. | Mensajes que representan respuestas previas del propio modelo. |

En Gemini, las instrucciones generales se colocan normalmente en `system_instruction` dentro de un objeto `GenerateContentConfig`, mientras que el contenido del usuario se pasa mediante `contents`.

En LangChain, disponemos de clases como `SystemMessage`, `HumanMessage` y `AIMessage`, o bien podemos pasar diccionarios con roles tipo `system`, `user`, `assistant` y `tool`.


In [ ]:
from google.genai import types as genai_types
from langchain_core.messages import SystemMessage, HumanMessage

# OpenAI: mensajes con roles
openai_role_response = openai_client.responses.create(
    model=OPENAI_MODEL,
    input=[
        {
            "role": "developer",
            "content": "Translate the following text to Italian."
        },
        {
            "role": "user",
            "content": "Hello, how are you?"
        }
    ],
)

print("OPENAI con roles:")
print(openai_role_response.output_text)

# Gemini: instrucción de sistema dentro de GenerateContentConfig
gemini_role_response = gemini_client.models.generate_content(
    model=GOOGLE_MODEL,
    config=genai_types.GenerateContentConfig(
        system_instruction="Translate the following text to Italian.",
    ),
    contents="Text: Hello, how are you?"
)

print("\nGEMINI con system_instruction:")
print(gemini_role_response.text)

# LangChain: clases de mensajes
langchain_role_response = langchain_openai_model.invoke([
    SystemMessage(content="Translate the following text to Italian."),
    HumanMessage(content="Hello, how are you?")
])

print("\nLANGCHAIN con clases de mensajes:")
print(langchain_role_response.content)


En OpenAI, el parámetro `instructions` nos permite expresar de forma compacta una instrucción de alto nivel. Para una tarea simple, el siguiente ejemplo es equivalente en espíritu al uso de un mensaje `developer` anterior.


In [ ]:
openai_instructions_response = openai_client.responses.create(
    model=OPENAI_MODEL,
    instructions="Translate the following text to Italian.",
    input="Hello, how are you?",
)

print(openai_instructions_response.output_text)


LangChain también admite listas de diccionarios con roles en lugar de clases explícitas. Esta forma resulta familiar si venimos de APIs que representan mensajes como estructuras JSON.

Los roles habituales en LangChain son:

| **Rol** | **Descripción** |
|---------|-----------------|
| `system` | Instrucciones generales y contexto de alto nivel. No todos los modelos soportan este tipo de rol del mismo modo. |
| `user` | Input del usuario final. |
| `assistant` | Respuesta previa del modelo. Puede ser texto o una solicitud de herramienta. |
| `tool` | Resultado de una función o herramienta externa invocada durante la ejecución. |


In [ ]:
langchain_dict_response = langchain_openai_model.invoke([
    {"role": "system", "content": "Translate the following text to Italian."},
    {"role": "user", "content": "Hello, how are you?"},
])

print(langchain_dict_response.content)


### 4.2. Parámetros de generación

Además del input, casi todas las APIs permiten controlar la generación mediante parámetros. No hace falta memorizarlos todos, pero sí conviene dominar los que aparecen constantemente.

En OpenAI, algunos parámetros habituales de `responses.create` son:

* `model`: identificador del modelo que se usará. Es obligatorio.
* `input`: texto, mensajes, imágenes o archivos que se suministran al modelo. Es obligatorio.
* `instructions`: instrucciones de alto nivel que se colocan al inicio del contexto del modelo.
* `previous_response_id`: ID de una respuesta anterior para continuar una conversación sin reenviar manualmente todo el historial.
* `max_output_tokens`: límite máximo de tokens que puede generar la respuesta.
* `temperature`: controla la aleatoriedad de la generación. Valores bajos suelen producir respuestas más deterministas; valores altos, respuestas más variadas.
* `text`: configuración de la salida textual, incluyendo formatos estructurados como JSON.
* `tools` y `tool_choice`: permiten habilitar herramientas externas o integradas.
* `stream`: permite recibir la respuesta progresivamente mediante eventos.
* `reasoning`: permite configurar el esfuerzo de razonamiento en modelos que lo soporten.
* `store`: controla si la respuesta queda almacenada para poder recuperarla o encadenarla posteriormente.

Para el detalle completo y actualizado, consulta la [referencia oficial de `responses.create`](https://platform.openai.com/docs/api-reference/responses/create).


En Gemini, muchos parámetros se agrupan en `GenerateContentConfig`. Entre los más relevantes encontramos:

* `stopSequences`: array de hasta 5 cadenas que indican al modelo dónde detener la generación.
* `responseMimeType`: tipo MIME del resultado devuelto. Los admitidos incluyen `text/plain`, `application/json` y `text/x.enum`.
* `responseSchema`: objeto que describe el esquema de la salida cuando se usa JSON.
* `responseModalities`: modalidades esperadas en la respuesta, por ejemplo texto o imagen, según lo soporte el modelo.
* `candidateCount`: número de respuestas candidatas a devolver.
* `maxOutputTokens`: límite máximo de tokens en cada candidato.
* `temperature`: controla la aleatoriedad del muestreo.
* `topP`: probabilidad acumulada máxima para el muestreo *nucleus*.
* `topK`: máximo número de tokens considerados en el muestreo Top-K.
* `seed`: semilla aleatoria para intentar hacer la generación más reproducible.
* `presencePenalty`: penaliza tokens ya aparecidos en la respuesta.
* `frequencyPenalty`: penaliza tokens según cuántas veces se han usado.
* `responseLogprobs`: indica si deben incluirse log-probs de la generación.
* `logprobs`: fija cuántos log-probs principales devolver.
* `enableEnhancedCivicAnswers`: activa un modo con respuestas cívicas mejoradas en modelos compatibles.
* `speechConfig`: configuración de síntesis de voz, cuando aplica.
* `thinkingConfig`: configuración de capacidades de pensamiento interno en modelos compatibles.
* `mediaResolution`: resolución del contenido multimedia generado, cuando aplica.

La nomenclatura suele alternar entre `camelCase` en documentación y `snake_case` en el SDK de Python. Por ejemplo, `responseMimeType` puede aparecer como `response_mime_type` en código Python.


In [ ]:
# Ejemplo de configuración comparable en OpenAI, Gemini y LangChain.
# No todos los parámetros tienen exactamente el mismo nombre ni la misma semántica entre proveedores.

openai_configured_response = openai_client.responses.create(
    model=OPENAI_MODEL,
    input="Give me one concise tip for learning Italian pronunciation.",
    max_output_tokens=80,
    temperature=0.2,
)
print("OPENAI:")
print(openai_configured_response.output_text)

gemini_configured_response = gemini_client.models.generate_content(
    model=GOOGLE_MODEL,
    contents="Give me one concise tip for learning Italian pronunciation.",
    config=genai_types.GenerateContentConfig(
        max_output_tokens=80,
        temperature=0.2,
    ),
)
print("\nGEMINI:")
print(gemini_configured_response.text)

langchain_configured_model = init_chat_model(
    OPENAI_MODEL,
    model_provider="openai",
    temperature=0.2,
    max_tokens=80,
)
langchain_configured_response = langchain_configured_model.invoke(
    "Give me one concise tip for learning Italian pronunciation."
)
print("\nLANGCHAIN + OPENAI:")
print(langchain_configured_response.content)


### 4.3. Prompt Templates en LangChain

Hasta ahora hemos pasado mensajes directamente al modelo. En producción, lo habitual es que el input del usuario no se envíe sin más: antes se combina con instrucciones de sistema, variables de aplicación, datos recuperados de una base de datos, idioma objetivo, formato de salida o restricciones de negocio.

Los [Prompt Templates](https://python.langchain.com/docs/concepts/prompt_templates/) son un concepto central en LangChain para representar esas transformaciones de forma reutilizable. Un template recibe variables y construye el prompt final que se enviará al modelo.

Supongamos que ahora no queremos traducir siempre a italiano, sino a cualquier idioma indicado dinámicamente.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "Translate the following from English to {language}."

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", system_template),
        ("user", "{text}"),
    ]
)

italian_prompt = prompt_template.invoke(
    {"language": "Italian", "text": "Hello, how are you?"}
)
spanish_prompt = prompt_template.invoke(
    {"language": "Spanish", "text": "Hello, how are you?"}
)

print("ITALIAN PROMPT:")
print(italian_prompt)
print("\nSPANISH PROMPT:")
print(spanish_prompt)


Los objetos generados por el template pueden convertirse en listas de mensajes mediante `to_messages`. Esto permite comprobar exactamente qué se enviará al modelo antes de ejecutar la llamada.


In [ ]:
print("Mensajes para italiano:")
print(italian_prompt.to_messages())

print("\nMensajes para español:")
print(spanish_prompt.to_messages())


In [ ]:
italian_response = langchain_openai_model.invoke(italian_prompt)
spanish_response = langchain_openai_model.invoke(spanish_prompt)

print("ITALIANO:", italian_response.content)
print("ESPAÑOL:", spanish_response.content)


## 5. Conversaciones multi-turno

Hasta ahora hemos hecho llamadas independientes. Pero muchas aplicaciones basadas en LLMs son conversacionales: el usuario pregunta algo, el modelo responde, el usuario hace una pregunta de seguimiento, y el modelo debería interpretar la nueva pregunta teniendo en cuenta lo anterior.

El punto clave es este: **los modelos no recuerdan mágicamente la conversación**. Una llamada de API no tiene memoria por defecto como la interfaz de ChatGPT. Si queremos que el modelo tenga contexto conversacional, debemos proporcionárselo de alguna forma.

Cada enfoque ofrece mecanismos distintos:

| Enfoque | Estrategia |
|---------|------------|
| OpenAI | Reenviar historial manualmente o usar `previous_response_id`. |
| Gemini | Usar `client.chats.create(...)`, que mantiene un historial de chat. |
| LangChain | Reenviar mensajes, usar `MessagesPlaceholder` o apoyarse en persistencia con LangGraph. |


### 5.1. Un prompt de asistente de traducción

Primero cargaremos un prompt más detallado desde la carpeta de recursos. La idea ya no será traducir cualquier cosa sin contexto, sino actuar como un asistente experto al que podemos hacer preguntas sobre traducción al italiano.


In [ ]:
with open("./resources/prompt_italian_buddy.txt") as f:
    italian_buddy_prompt = f.read()

first_question_text = "How do you say 'Hello, how are you?' in Italian?"
follow_up_question_text = "But the first word isn't like 'Goodbye'?"

print(italian_buddy_prompt[:500])


### 5.2. Primera interacción en los tres enfoques

Empecemos con una primera pregunta. Aquí todavía no hay historial previo, por lo que la llamada es sencilla.


In [ ]:
openai_first_response = openai_client.responses.create(
    model=OPENAI_MODEL,
    instructions=italian_buddy_prompt,
    input=first_question_text,
)
openai_first_answer = openai_first_response.output_text

print("OPENAI:")
print(openai_first_answer)

gemini_first_response = gemini_client.models.generate_content(
    model=GOOGLE_MODEL,
    config=genai_types.GenerateContentConfig(
        system_instruction=italian_buddy_prompt,
    ),
    contents=first_question_text,
)
gemini_first_answer = gemini_first_response.text

print("\nGEMINI:")
print(gemini_first_answer)

langchain_first_response = langchain_openai_model.invoke([
    SystemMessage(content=italian_buddy_prompt),
    HumanMessage(content=first_question_text),
])
langchain_first_answer = langchain_first_response.content

print("\nLANGCHAIN + OPENAI:")
print(langchain_first_answer)


### 5.3. Qué ocurre si hacemos una pregunta de seguimiento sin historial

Ahora hagamos una pregunta de seguimiento: *But the first word isn't like 'Goodbye'?*.

Para una persona, la pregunta hace referencia a la respuesta anterior. Para una llamada aislada al modelo, esa referencia puede no estar clara. Si no enviamos el historial, el modelo solo ve la nueva frase y las instrucciones de sistema.


In [ ]:
openai_no_history_response = openai_client.responses.create(
    model=OPENAI_MODEL,
    instructions=italian_buddy_prompt,
    input=follow_up_question_text,
)

print("OPENAI sin historial:")
print(openai_no_history_response.output_text)

gemini_no_history_response = gemini_client.models.generate_content(
    model=GOOGLE_MODEL,
    config=genai_types.GenerateContentConfig(
        system_instruction=italian_buddy_prompt,
    ),
    contents=follow_up_question_text,
)

print("\nGEMINI sin historial:")
print(gemini_no_history_response.text)

langchain_no_history_response = langchain_openai_model.invoke([
    SystemMessage(content=italian_buddy_prompt),
    HumanMessage(content=follow_up_question_text),
])

print("\nLANGCHAIN sin historial:")
print(langchain_no_history_response.content)


El comportamiento exacto puede variar, pero conceptualmente el problema es el mismo: el modelo no tiene por qué saber a qué palabra se refiere el usuario. Para resolverlo, debemos enviar el contexto de la conversación.


### 5.4. Historial manual en OpenAI

La forma más explícita consiste en reconstruir la conversación como una lista de mensajes. Incluimos la primera pregunta del usuario, la respuesta del asistente y la pregunta de seguimiento. Así el modelo puede interpretar la segunda pregunta dentro de su contexto.


In [ ]:
openai_manual_final_response = openai_client.responses.create(
    model=OPENAI_MODEL,
    instructions=italian_buddy_prompt,
    input=[
        {"role": "user", "content": first_question_text},
        {"role": "assistant", "content": openai_first_answer},
        {"role": "user", "content": follow_up_question_text},
    ],
)
openai_manual_final_answer = openai_manual_final_response.output_text

print("CONVERSATION:")
print(f"\t- User: {first_question_text}")
print(f"\t- Assistant: {openai_first_answer}")
print(f"\t- User: {follow_up_question_text}")
print(f"\t- Assistant: {openai_manual_final_answer}")


### 5.5. Continuidad con `previous_response_id` en OpenAI

OpenAI también ofrece una opción más cómoda para continuar conversaciones: `previous_response_id`. En lugar de reenviar manualmente todo el historial, podemos indicar que una respuesta continúa a partir de una respuesta previa.

Esto simplifica la construcción de interfaces conversacionales, aunque sigue siendo importante entender que el contexto tiene coste y ocupa ventana de contexto. El hecho de que el SDK nos facilite la continuidad no significa que el contexto sea gratuito ni infinito.


In [ ]:
openai_previous_id_final_response = openai_client.responses.create(
    model=OPENAI_MODEL,
    instructions=italian_buddy_prompt,
    input=follow_up_question_text,
    previous_response_id=openai_first_response.id,
)
openai_previous_id_final_answer = openai_previous_id_final_response.output_text

print("CONVERSATION:")
print(f"\t- User: {first_question_text}")
print(f"\t- Assistant: {openai_first_answer}")
print(f"\t- User: {follow_up_question_text}")
print(f"\t- Assistant: {openai_previous_id_final_answer}")


### 5.6. Conversación multi-turno en Gemini con `Chat`

En Gemini podemos crear una instancia de chat mediante `client.chats.create`. Esta clase se encarga de mantener el historial por nosotros durante la vida del objeto. A partir de ahí, enviamos mensajes con `send_message`.

La función `send_message` también acepta un parámetro opcional `config`, donde podemos enviar un objeto `GenerateContentConfig` para modificar configuraciones de generación.


In [ ]:
gemini_chat = gemini_client.chats.create(model=GOOGLE_MODEL)

gemini_chat_first_response = gemini_chat.send_message(first_question_text)
gemini_chat_final_response = gemini_chat.send_message(follow_up_question_text)

print("CONVERSATION:")
for message in gemini_chat.get_history():
    print(f"\n[{message.role.title()}]", end=": ")
    print(message.parts[0].text)


In [ ]:
type(gemini_chat)


Podemos acceder al historial completo de la conversación a través del método `get_history`. Esta abstracción resulta cómoda para prototipos y aplicaciones conversacionales sencillas, porque evita construir manualmente la lista completa de mensajes en cada turno.


### 5.7. Historial manual en LangChain con `MessagesPlaceholder`

En LangChain también podemos gestionar conversaciones pasando explícitamente la lista completa de mensajes. Si queremos crear un prompt template con un número variable de mensajes, utilizamos `MessagesPlaceholder`.

`MessagesPlaceholder` actúa como una ranura donde LangChain insertará la conversación actual: uno, dos, diez o cien mensajes, según lo que le pasemos en cada invocación.


In [ ]:
from langchain_core.prompts import MessagesPlaceholder

conversation_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", italian_buddy_prompt),
        MessagesPlaceholder("msgs"),
    ]
)

langchain_first_question = HumanMessage(content=first_question_text)
langchain_first_prompt = conversation_prompt_template.invoke(
    {"msgs": [langchain_first_question]}
)
langchain_first_answer = langchain_openai_model.invoke(langchain_first_prompt)

langchain_follow_up_question = HumanMessage(content=follow_up_question_text)
langchain_final_prompt = conversation_prompt_template.invoke(
    {"msgs": [langchain_first_question, langchain_first_answer, langchain_follow_up_question]}
)
langchain_final_answer = langchain_openai_model.invoke(langchain_final_prompt)

print("CONVERSATION:")
print(f"User: {langchain_first_question.content}")
print(f"Assistant: {langchain_first_answer.content}")
print(f"User: {langchain_follow_up_question.content}")
print(f"Assistant: {langchain_final_answer.content}")


### 5.8. Persistencia conversacional con LangGraph

LangChain puede gestionar prompts y mensajes, pero cuando queremos memoria persistente de conversaciones suele entrar en juego LangGraph. LangGraph permite definir grafos de ejecución y añadir *checkpointers* para conservar el estado entre llamadas.

Aquí construiremos un grafo mínimo:

1. El estado contiene una lista de mensajes.
2. Un nodo `model` añade el prompt de sistema y llama al modelo.
3. `MemorySaver` conserva el historial por `thread_id`.
4. Dos llamadas con el mismo `thread_id` comparten contexto.


In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

workflow = StateGraph(state_schema=MessagesState)

# Define la función que llama al modelo
def call_model(state: MessagesState):
    messages = [SystemMessage(content=italian_buddy_prompt)] + state["messages"]
    response = langchain_openai_model.invoke(messages)
    return {"messages": response}

workflow.add_node("model", call_model)
workflow.add_edge(START, "model")

memory = MemorySaver()
conversation_app = workflow.compile(checkpointer=memory)


In [ ]:
conversation_app.invoke(
    {"messages": [HumanMessage(content="How do you say 'Hello, how are you?' in Italian?")]},
    config={"configurable": {"thread_id": 1}},
)


In [ ]:
conversation_app.invoke(
    {"messages": [HumanMessage(content="But the first word isn't like 'Goodbye'?")]},
    config={"configurable": {"thread_id": 1}},
)


Si cambiamos el `thread_id`, LangGraph tratará la llamada como otra conversación. Este patrón es fundamental para aplicaciones multiusuario: cada usuario, sesión o conversación necesita su propio identificador de hilo.


## 6. Ventana de contexto, tokens y coste

La **ventana de contexto** es el número máximo de tokens que un modelo puede gestionar en una sola llamada. Incluye el input, las instrucciones, el historial, los documentos adjuntos, las llamadas a herramientas y la respuesta generada. En modelos que usan razonamiento interno, también puede haber tokens asociados a ese razonamiento.

Esto tiene tres consecuencias prácticas:

* **Coste:** si reenviamos mucho historial, pagamos más tokens de entrada.
* **Latencia:** más tokens suelen implicar más tiempo de procesamiento.
* **Límites:** si superamos la ventana de contexto, la llamada fallará o tendremos que recortar información.

<img src="./img/openai_ventana_contexto.png" alt="Gestión de la ventana de contexto en OpenAI" style="display:block; margin:auto; margin-bottom:20px;" width="500"/>

> El número máximo de tokens disponibles depende del modelo. También existe una limitación sobre el número máximo de tokens que se pueden generar en el output.


### 6.1. Uso real de tokens en OpenAI

Los objetos de respuesta de OpenAI incluyen información de uso. Podemos observar tokens de input, tokens cacheados, tokens de output, tokens de razonamiento y total de tokens.

La caché puede abaratar llamadas cuando parte del contexto se reutiliza, pero no debemos confundirla con memoria gratuita. El contexto sigue ocupando ventana de contexto y sigue formando parte de la interacción.


In [ ]:
print("PRIMERA INTERACCIÓN:")
print("Input tokens:", openai_first_response.usage.input_tokens)
print("Input tokens (cached):", openai_first_response.usage.input_tokens_details.cached_tokens)
print("Output tokens:", openai_first_response.usage.output_tokens)
print("Reasoning tokens:", openai_first_response.usage.output_tokens_details.reasoning_tokens)
print("Total tokens:", openai_first_response.usage.total_tokens)
print("--------------------")
print("SEGUNDA INTERACCIÓN:")
print("Input tokens:", openai_previous_id_final_response.usage.input_tokens)
print("Input tokens (cached):", openai_previous_id_final_response.usage.input_tokens_details.cached_tokens)
print("Output tokens:", openai_previous_id_final_response.usage.output_tokens)
print("Reasoning tokens:", openai_previous_id_final_response.usage.output_tokens_details.reasoning_tokens)
print("Total tokens:", openai_previous_id_final_response.usage.total_tokens)


### 6.2. Estimación de tokens con `tiktoken`

También podemos estimar cuántos tokens representa un texto usando `tiktoken`. Esta estimación es útil para controles previos, validaciones de tamaño, recorte de documentos o cálculos aproximados de coste.

Aun así, el número de tokens calculado localmente puede no coincidir exactamente con el que reporta la API, porque la llamada final puede incluir serialización interna, mensajes de sistema, wrappers o detalles del proveedor.


In [ ]:
import tiktoken

try:
    enc = tiktoken.encoding_for_model(OPENAI_MODEL)
except KeyError:
    enc = tiktoken.get_encoding("o200k_base")

tokens_system_prompt = len(enc.encode(italian_buddy_prompt))
tokens_first_question = len(enc.encode(first_question_text))
tokens_first_answer = len(enc.encode(openai_first_answer))
tokens_follow_up_question = len(enc.encode(follow_up_question_text))
tokens_final_answer = len(enc.encode(openai_previous_id_final_answer))

print("PRIMERA INTERACCIÓN:")
print("System tokens:", tokens_system_prompt)
print("Question tokens:", tokens_first_question)
print("Input tokens:", tokens_system_prompt + tokens_first_question)
print("Output tokens:", tokens_first_answer)
print("Total tokens:", tokens_system_prompt + tokens_first_question + tokens_first_answer)
print("--------------------")
print("SEGUNDA INTERACCIÓN:")
print("System tokens:", tokens_system_prompt)
print("Follow-up question tokens:", tokens_follow_up_question)
print("Input tokens:", tokens_system_prompt + tokens_first_question + tokens_first_answer + tokens_follow_up_question)
print("Output tokens:", tokens_final_answer)
print("Total tokens:", tokens_system_prompt + tokens_first_question + tokens_first_answer + tokens_follow_up_question + tokens_final_answer)


Nota que existen discrepancias entre el número de tokens proporcionado por la API y los calculados por `tiktoken`. Usa `tiktoken` como una estimación, no como la verdad absoluta de facturación.

Para ver cómo funciona el tokenizador, puedes usar la herramienta interactiva [Tokenizer](https://platform.openai.com/tokenizer).


### 6.3. Ventana de contexto y tokens en Gemini

Un aspecto muy positivo de algunos modelos de Google es su gran ventana de contexto. Google ha ofrecido variantes de Gemini con ventanas de contexto de hasta 1 millón de tokens. Según Google, ese orden de magnitud puede representar aproximadamente:

* 50.000 líneas de código con 80 caracteres por línea.
* Todos los mensajes de texto que has enviado en los últimos 5 años.
* 8 novelas completas en inglés de longitud promedio.
* La transcripción de 200 podcasts completos de duración promedio.

Nótese que también puede haber límites distintos para tokens de salida.

<img src="./img/google_tokens_gemini3.png" alt="Tokens de Gemini 3.1 Pro" style="display:block; margin:auto; margin-bottom:20px;" width="500"/>

Podemos consultar límites de variantes concretas mediante `client.models.get`.


In [ ]:
print("GEMINI 3.1 PRO PREVIEW:")
model_info = gemini_client.models.get(model="gemini-3.1-pro-preview")
print(f"{model_info.input_token_limit = :,.0f}")
print(f"{model_info.output_token_limit = :,.0f}")

print("\nGEMINI 3 FLASH:")
model_info = gemini_client.models.get(model="gemini-3-flash-preview")
print(f"{model_info.input_token_limit = :,.0f}")
print(f"{model_info.output_token_limit = :,.0f}")

print("\nGEMINI 3.1 FLASH LITE:")
model_info = gemini_client.models.get(model="gemini-3.1-flash-lite")
print(f"{model_info.input_token_limit = :,.0f}")
print(f"{model_info.output_token_limit = :,.0f}")


Gemini también ofrece una función para contar tokens antes de llamar al modelo. Esto permite hacer una gestión más inteligente del contexto: decidir si podemos enviar un documento completo, si tenemos que resumirlo, si conviene trocearlo o si debemos rechazar una entrada demasiado grande.


In [ ]:
question = "How do you say 'Hello, how are you?' in Italian?"

print("TOTAL TOKENS:")

tokens = gemini_client.models.count_tokens(
    model=GOOGLE_MODEL,
    contents=question,
)
print("\t- Estimated before calling the model:", tokens.total_tokens)

gemini_token_response = gemini_client.models.generate_content(
    model=GOOGLE_MODEL,
    contents=question,
)

print("\t- Actual input tokens used:", gemini_token_response.usage_metadata.prompt_token_count)


### 6.4. Tokens desde LangChain

LangChain no elimina la realidad de los tokens. Si usamos un modelo de OpenAI, Google u otro proveedor a través de LangChain, el coste y los límites siguen siendo los del proveedor subyacente.

Dependiendo del wrapper, la respuesta puede incluir `usage_metadata`. Esto resulta cómodo para instrumentar aplicaciones sin depender directamente del SDK nativo, aunque no todos los proveedores devuelven exactamente la misma estructura.


In [ ]:
langchain_usage_response = langchain_openai_model.invoke(
    "Give me a short explanation of what a context window is."
)

print("LANGCHAIN response content:")
print(langchain_usage_response.content)
print("\nUsage metadata:")
print(getattr(langchain_usage_response, "usage_metadata", None))


## 7. Streaming de respuestas

Hasta ahora hemos esperado a que la respuesta completa esté generada antes de mostrarla. Sin embargo, en interfaces conversacionales modernas estamos acostumbrados a ver cómo el texto aparece progresivamente.

Esta técnica se llama **streaming**. En vez de recibir una única respuesta final, recibimos eventos o fragmentos parciales que podemos ir mostrando al usuario.

El streaming mejora la experiencia percibida cuando:

* La respuesta es larga.
* El modelo tarda varios segundos en completar la generación.
* Queremos que el usuario empiece a leer antes de que termine toda la respuesta.
* Construimos una interfaz tipo chat.


### 7.1. Streaming en OpenAI

En OpenAI activamos el streaming con `stream=True`. La llamada devuelve un iterador de eventos. Estos eventos no son todos texto: algunos indican que la respuesta se ha creado, que está en progreso, que se completó, que se añadió un item, que se abrió una llamada a herramienta, etc.


In [ ]:
openai_stream = openai_client.responses.create(
    model=OPENAI_MODEL,
    input="Write an original short Italian song about learning a language.",
    stream=True,
)

for event in openai_stream:
    print(event)


Entre los tipos de eventos que podemos encontrar están:

* `ResponseCreatedEvent`
* `ResponseInProgressEvent`
* `ResponseFailedEvent`
* `ResponseCompletedEvent`
* `ResponseOutputItemAdded`
* `ResponseOutputItemDone`
* `ResponseContentPartAdded`
* `ResponseContentPartDone`
* `ResponseOutputTextDelta`
* `ResponseOutputTextAnnotationAdded`
* `ResponseTextDone`
* `ResponseRefusalDelta`
* `ResponseRefusalDone`
* `ResponseFunctionCallArgumentsDelta`
* `ResponseFunctionCallArgumentsDone`
* `ResponseFileSearchCallInProgress`
* `ResponseFileSearchCallSearching`
* `ResponseFileSearchCallCompleted`
* `ResponseCodeInterpreterInProgress`
* `ResponseCodeInterpreterCallCodeDelta`
* `ResponseCodeInterpreterCallCodeDone`
* `ResponseCodeInterpreterCallIntepreting`
* `ResponseCodeInterpreterCallCompleted`
* `Error`

Para mostrar texto de forma progresiva, filtraremos los eventos de tipo `response.output_text.delta`.


In [ ]:
import time

openai_stream = openai_client.responses.create(
    model=OPENAI_MODEL,
    input="Write an original short Italian song about learning a language.",
    stream=True,
)

for event in openai_stream:
    if event.type == "response.output_text.delta":
        time.sleep(0.3)
        print(event.delta, end="")


### 7.2. Streaming en Gemini

En Gemini no activamos un booleano dentro de la misma función, sino que llamamos a `generate_content_stream`. El resultado también es iterable, pero en este caso recibimos chunks de contenido.


In [ ]:
gemini_stream = gemini_client.models.generate_content_stream(
    model=GOOGLE_MODEL,
    contents="Explain how to make a perfect Italian pizza.",
)

for chunk in gemini_stream:
    print(chunk.text, end="")


### 7.3. Streaming en LangChain

LangChain expone una interfaz común para streaming mediante `.stream(...)`. Cada fragmento suele ser un `AIMessageChunk`, análogo a un `AIMessage` pero parcial.

Esto es especialmente útil si queremos construir una interfaz que pueda cambiar de proveedor sin reescribir por completo el mecanismo de streaming.


In [ ]:
print("LANGCHAIN + OPENAI:\n")
for chunk in langchain_openai_model.stream(
    "Write an original short Italian song about learning a language."
):
    print(chunk.content, end="")

print("\n\nLANGCHAIN + GEMINI:\n")
for chunk in langchain_google_model.stream(
    "Explain how to make a perfect Italian pizza."
):
    print(chunk.content, end="")


## 8. Salidas estructuradas en JSON

Cuando integramos un LLM en una aplicación real, muchas veces no queremos solo texto libre. Queremos un objeto que podamos validar y procesar desde código.

Ejemplos:

* Una traducción con metadatos.
* Una clasificación con etiqueta y confianza.
* Una extracción de entidades.
* Una respuesta preparada para guardarse en base de datos.
* Un payload que alimentará otra API.

Pedir simplemente “devuélveme JSON” en el prompt es frágil. Lo más robusto es definir un esquema de salida y pedir al proveedor o framework que lo respete.


### 8.1. Esquema común de traducción

Usaremos el mismo contrato de datos en los tres enfoques. Queremos recibir:

* `translation`: mejor traducción al italiano.
* `register`: registro de la traducción (`formal`, `informal` o `neutral`).
* `alternatives`: alternativas posibles.

En APIs directas lo expresaremos como esquema JSON. En LangChain lo expresaremos como modelo Pydantic.


In [ ]:
import json

translation_schema = {
    "type": "object",
    "properties": {
        "translation": {"type": "string"},
        "register": {"type": "string", "enum": ["formal", "informal", "neutral"]},
        "alternatives": {
            "type": "array",
            "items": {"type": "string"},
        },
    },
    "required": ["translation", "register", "alternatives"],
    "additionalProperties": False,
}


### 8.2. JSON estructurado en OpenAI

En OpenAI podemos usar `text.format` con `json_schema`. Si activamos `strict=True`, estamos pidiendo que la salida siga estrictamente el esquema definido.


In [ ]:
openai_structured_response = openai_client.responses.create(
    model=OPENAI_MODEL,
    input="Translate 'Hello, how are you?' into Italian and explain the register briefly.",
    text={
        "format": {
            "type": "json_schema",
            "name": "translation_result",
            "schema": translation_schema,
            "strict": True,
        }
    },
)

openai_structured_object = json.loads(openai_structured_response.output_text)
openai_structured_object


### 8.3. JSON estructurado en Gemini

En Gemini configuramos el tipo MIME de respuesta como `application/json` y pasamos el esquema mediante `response_schema`.

El patrón conceptual es el mismo: definimos el formato esperado y parseamos el texto devuelto como JSON.


In [ ]:
translation_schema = {
    "type": "object",
    "properties": {
        "translation": {"type": "string"},
        "register": {"type": "string", "enum": ["formal", "informal", "neutral"]},
        "alternatives": {
            "type": "array",
            "items": {"type": "string"},
        },
    },
    "required": ["translation", "register", "alternatives"],
    # "additionalProperties": False,
}

In [ ]:
gemini_structured_response = gemini_client.models.generate_content(
    model=GOOGLE_MODEL,
    contents="Translate 'Hello, how are you?' into Italian and return two alternatives.",
    config=genai_types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=translation_schema,
    ),
)

gemini_structured_object = json.loads(gemini_structured_response.text)
gemini_structured_object


### 8.4. Salida estructurada en LangChain con Pydantic

LangChain permite pedir salidas estructuradas mediante `with_structured_output`. La ventaja es que trabajamos con un esquema de Python y LangChain se encarga de traducirlo a la forma que espera el proveedor.

Esto ayuda a mantener el código más agnóstico: el mismo patrón puede usarse con modelos de OpenAI, Google u otros proveedores soportados, siempre que el modelo elegido admita salida estructurada.


In [ ]:
from pydantic import BaseModel, Field

class TranslationResult(BaseModel):
    translation: str = Field(description="Best Italian translation")
    register: str = Field(description="Register of the translation: formal, informal or neutral")
    alternatives: list[str] = Field(description="Two alternative translations")

structured_openai_model = langchain_openai_model.with_structured_output(TranslationResult)
structured_openai_result = structured_openai_model.invoke(
    "Translate 'Hello, how are you?' into Italian and provide two alternatives."
)

structured_openai_result


In [ ]:
structured_google_model = langchain_google_model.with_structured_output(TranslationResult)
structured_google_result = structured_google_model.invoke(
    "Translate 'Hello, how are you?' into Italian and provide two alternatives."
)

structured_google_result


## 9. Qué aporta LangChain como capa de abstracción

Después de haber visto OpenAI y Gemini directamente, LangChain debería verse como una capa de orquestación, no como un proveedor más.

LangChain aporta:

* **Wrappers de modelos:** una interfaz común para modelos de distintos proveedores.
* **Mensajes normalizados:** `SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage` y `AIMessageChunk`.
* **Prompt Templates:** composición reutilizable de prompts con variables.
* **Chains:** secuencias fijas de pasos que conectan prompts, modelos, parsers, herramientas y otros componentes.
* **Output Parsers y salida estructurada:** conversión de texto o respuestas del modelo a objetos validados.
* **Integración con LangGraph:** flujos con estado, persistencia, memoria conversacional y grafos de ejecución.

La abstracción permite escribir menos código específico de proveedor, pero no elimina las diferencias reales entre modelos: límites, precios, soporte de roles, streaming, tool calling, formatos estructurados y metadatos pueden variar.


## 10. Recapitulación

En esta sesión hemos pasado de usar un modelo entrenado para una tarea concreta a consumir modelos fundacionales ya entrenados mediante APIs. Esa diferencia es importante: ahora el reto principal no es entrenar el modelo, sino saber elegirlo, configurarlo, darle instrucciones, gestionar el contexto y convertir sus respuestas en algo útil para una aplicación.

Ideas clave de la sesión:

* Las APIs de OpenAI y Google siguen patrones muy parecidos: cliente, modelo, input, configuración y respuesta.
* LangChain permite trabajar con distintos proveedores mediante wrappers, mensajes y templates comunes.
* Los roles o instrucciones de sistema permiten separar la lógica de la aplicación del input del usuario.
* Los modelos no recuerdan mágicamente la conversación: hay que reenviar historial, usar mecanismos como `previous_response_id`, apoyarse en objetos `Chat` o utilizar persistencia con LangGraph.
* La ventana de contexto y los tokens afectan al coste, a la latencia y a la posibilidad de que una respuesta quede truncada.
* El streaming mejora la experiencia de usuario cuando la respuesta tarda en generarse.
* Las salidas estructuradas en JSON son imprescindibles cuando el LLM se integra en software real.
* La abstracción no elimina los fundamentos: aunque cambie la librería, seguimos trabajando con modelos, mensajes, instrucciones, contexto, tokens, streaming, herramientas y estructura de salida.

La idea práctica no es casarse con un proveedor ni con una herramienta concreta, sino entender los conceptos que se repiten entre proveedores y saber trasladarlos de una sintaxis a otra.
